# EPEN R3 · Qwen en GPU de Colab

Este cuaderno levanta un servicio **temporal** compatible con el dashboard. Solo debe usarse con datos sintéticos o expresamente autorizados. El token y la URL cambian en cada sesión. No compartas una copia del notebook que conserve salidas con secretos.

Antes de empezar: **Entorno de ejecución → Cambiar tipo de entorno → GPU**.

In [ ]:
import subprocess
gpu = subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], text=True).strip()
print('GPU asignada:', gpu)
if not gpu:
    raise RuntimeError('Colab no asignó una GPU. Cambia el tipo de entorno antes de continuar.')

In [ ]:
# Instala Ollama y cloudflared únicamente dentro de la VM efímera de Colab.
import shutil, subprocess
subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'zstd', 'pciutils'], check=True)
subprocess.run('curl -fsSL https://ollama.com/install.sh | sh', shell=True, check=True)
subprocess.run(['wget', '-q', 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', '-O', '/usr/local/bin/cloudflared'], check=True)
subprocess.run(['chmod', '+x', '/usr/local/bin/cloudflared'], check=True)
if not shutil.which('ollama'):
    raise RuntimeError('Ollama no quedó instalado; revisa la salida anterior.')
if not shutil.which('cloudflared'):
    raise RuntimeError('cloudflared no quedó instalado.')
print('Dependencias verificadas:', shutil.which('ollama'), shutil.which('cloudflared'))

In [ ]:
# Descarga el servicio o reutiliza/normaliza el archivo ya subido en esta VM.
from pathlib import Path
from urllib.request import urlretrieve
service_path = Path('/content/r3_gpu_service.py')
raw_url = 'https://raw.githubusercontent.com/RoyPavelSolierZamora/proyecto_integrador_inei_2026/main/colab/r3_gpu_service.py'
if service_path.exists():
    print('Se reutilizará /content/r3_gpu_service.py.')
else:
    try:
        urlretrieve(raw_url, service_path)
        print('Servicio descargado del repositorio.')
    except Exception:
        from google.colab import files
        print('Sube colab/r3_gpu_service.py desde la carpeta del proyecto.')
        uploaded = files.upload()
        python_files = [(name, content) for name, content in uploaded.items() if name.lower().endswith('.py')]
        if len(python_files) != 1:
            raise RuntimeError('Selecciona exactamente un archivo Python: r3_gpu_service.py.')
        uploaded_name, content = python_files[0]
        service_path.write_bytes(content)
        print('Archivo normalizado:', uploaded_name, '->', service_path)
print('SHA local:', __import__('hashlib').sha256(service_path.read_bytes()).hexdigest()[:12])

In [ ]:
import os, secrets, subprocess, time, urllib.request, json
MODEL = 'qwen3:4b-instruct-2507-q4_K_M'
TOKEN = secrets.token_urlsafe(32)
os.environ['QWEN_MODEL'] = MODEL
os.environ['R3_API_TOKEN'] = TOKEN

try:
    urllib.request.urlopen('http://127.0.0.1:11434/api/tags', timeout=2)
    OLLAMA_PROCESS = None
    print('Se reutilizará Ollama ya activo.')
except Exception:
    ollama_log = open('/content/ollama.log', 'w', encoding='utf-8')
    OLLAMA_PROCESS = subprocess.Popen(['ollama', 'serve'], stdout=ollama_log, stderr=subprocess.STDOUT, env=os.environ.copy())
    for _ in range(40):
        try:
            urllib.request.urlopen('http://127.0.0.1:11434/api/tags', timeout=2)
            break
        except Exception:
            time.sleep(1)
    else:
        raise RuntimeError('Ollama no inició. Revisa /content/ollama.log')
subprocess.run(['ollama', 'pull', MODEL], check=True)
# El puerto 8000 pertenece exclusivamente a R3 dentro de esta VM efímera.
subprocess.run(['fuser', '-k', '8000/tcp'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=False)
time.sleep(1)
r3_log = open('/content/r3.log', 'w', encoding='utf-8')
R3_PROCESS = subprocess.Popen(['python', str(service_path)], stdout=r3_log, stderr=subprocess.STDOUT, env=os.environ.copy())
time.sleep(3)
if R3_PROCESS.poll() is not None:
    raise RuntimeError(Path('/content/r3.log').read_text(encoding='utf-8'))
print('Ollama, modelo y R3 iniciados.')

In [ ]:
import re, threading, queue
previous_tunnel = globals().get('TUNNEL_PROCESS')
if previous_tunnel and previous_tunnel.poll() is None:
    previous_tunnel.terminate()
    time.sleep(1)
TUNNEL_PROCESS = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000', '--no-autoupdate'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
)
lines = queue.Queue()
threading.Thread(target=lambda: [lines.put(line) for line in TUNNEL_PROCESS.stdout], daemon=True).start()
PUBLIC_URL = None
deadline = time.time() + 45
while time.time() < deadline and not PUBLIC_URL:
    try:
        line = lines.get(timeout=1)
        match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
        if match:
            PUBLIC_URL = match.group(0)
    except queue.Empty:
        pass
if not PUBLIC_URL:
    raise RuntimeError('No se obtuvo URL del túnel Cloudflare.')
# Valida R3 dentro de Colab. La URL pública se valida desde el backend local,
# porque algunas VM de Colab no resuelven inmediatamente su propio trycloudflare.
local_request = urllib.request.Request('http://127.0.0.1:8000/salud', headers={'Authorization': 'Bearer ' + TOKEN})
with urllib.request.urlopen(local_request, timeout=20) as response:
    HEALTH = json.loads(response.read().decode('utf-8'))
print('Salud:', HEALTH)
print()
print('Copia en el .env local y reinicia el dashboard:')
print()
print('R3_MODE=remote')
print('R3_BASE_URL=' + PUBLIC_URL)
print('R3_API_TOKEN=' + TOKEN)
print('QWEN_MODEL=' + MODEL)
print()
print('No compartas este token. Borra la salida antes de guardar o compartir el notebook.')

In [ ]:
# Prueba sintética: mide latencia y tokens/s sin enviar datos reales.
sample = {
    'id_registro': 'SINTETICO-COLAB-001',
    'p309': 'Venta de almuerzos preparados',
    'p308': 'Cocinera',
    'p308a': 'Prepara y vende menús',
    'candidatos': [
        {'codigo': '5610', 'descripcion': 'Actividades de restaurantes y servicio móvil de comidas'},
        {'codigo': '5629', 'descripcion': 'Otras actividades de servicio de comidas'},
        {'codigo': '4781', 'descripcion': 'Venta al por menor de alimentos en puestos de venta'},
    ],
    'casos_similares': []
}
payload = json.dumps(sample).encode('utf-8')
request = urllib.request.Request(PUBLIC_URL + '/codificar', data=payload, method='POST', headers={
    'Authorization': 'Bearer ' + TOKEN, 'Content-Type': 'application/json', 'Accept': 'application/json'
})
with urllib.request.urlopen(request, timeout=180) as response:
    result = json.loads(response.read().decode('utf-8'))
print(json.dumps(result, ensure_ascii=False, indent=2))

## Finalizar correctamente

Cuando termines, ejecuta la siguiente celda. Después cambia el dashboard nuevamente a `R3_MODE=local` si no continuarás usando Colab.

In [ ]:
for process in [globals().get('TUNNEL_PROCESS'), globals().get('R3_PROCESS'), globals().get('OLLAMA_PROCESS')]:
    if process and process.poll() is None:
        process.terminate()
print('Servicios temporales detenidos. El token anterior dejó de ser utilizable.')